<a href="https://colab.research.google.com/github/Gcanales1548/CHATBOT-CHALENGE-ALURA/blob/main/chatbot_soporte_control_personal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CHATBOT RAG DE SOPORTE PARA CONTROL PERSONAL
# CÓDIGO COMPLETO PARA GOOGLE COLAB
# ============================================================
#
# Este programa:
#
# 1. Instala todas las librerías necesarias.
# 2. Obtiene la API Key de Gemini desde los secretos de Colab.
# 3. Detecta automáticamente un modelo Gemini disponible.
# 4. Permite subir documentos Word y PDF.
# 5. Extrae y limpia el contenido de los documentos.
# 6. Divide los documentos en fragmentos.
# 7. Crea embeddings locales con Hugging Face.
# 8. Guarda los fragmentos en una base vectorial FAISS.
# 9. Recupera información relacionada con cada pregunta.
# 10. Genera respuestas mediante Gemini.
# 11. Muestra las fuentes documentales consultadas.
# 12. Abre una interfaz web con Gradio.
#
# IMPORTANTE:
#
# Antes de ejecutar este código debes crear un secreto en Colab:
#
# Nombre: GOOGLE_API_KEY
# Valor: tu clave obtenida desde Google AI Studio
#
# En Colab:
# Barra lateral izquierda -> icono de llave -> Añadir secreto.
#
# ============================================================


# ============================================================
# 1. INSTALAR LIBRERÍAS
# ============================================================

# Instalamos las versiones actuales de las dependencias.
# La opción -q reduce los mensajes de instalación.

!pip install -q -U \
    google-genai \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    docx2txt \
    pypdf \
    gradio


# ============================================================
# 2. IMPORTAR LIBRERÍAS
# ============================================================

import os
import re
import shutil
import traceback
import warnings

from pathlib import Path
from typing import List, Tuple, Optional, Any

import gradio as gr

from google import genai
from google.genai import types
from google.colab import files, userdata

from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyPDFLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    FAISS
)

from langchain_core.documents import (
    Document
)


# Ocultamos advertencias menores para mantener limpio el notebook.
warnings.filterwarnings("ignore")

print("=" * 70)
print("CHATBOT DE SOPORTE — CONTROL PERSONAL")
print("=" * 70)
print()
print("Librerías importadas correctamente.")
print("Versión de Gradio:", gr.__version__)


# ============================================================
# 3. CONFIGURACIÓN GENERAL
# ============================================================

# Tamaño aproximado de cada fragmento documental.
TAMANO_FRAGMENTO = 1000

# Cantidad de caracteres repetidos entre fragmentos.
SOLAPAMIENTO_FRAGMENTOS = 200

# Cantidad de fragmentos recuperados para responder.
CANTIDAD_RESULTADOS = 5

# Distancia máxima aceptable de FAISS.
#
# En FAISS, un valor menor significa mayor similitud.
# Este valor puede ajustarse si el chatbot rechaza demasiadas
# preguntas correctas o responde preguntas poco relacionadas.
UMBRAL_DISTANCIA = 1.45

# Modelo local para crear embeddings.
#
# Este modelo es multilingüe y trabaja correctamente en español.
MODELO_EMBEDDINGS_LOCAL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# Carpeta donde se guardarán los documentos.
CARPETA_DOCUMENTOS = Path(
    "/content/documentos_control_personal"
)

# Carpeta donde se guardará la base FAISS.
CARPETA_FAISS = Path(
    "/content/base_vectorial_control_personal"
)

# Creamos las carpetas necesarias.
CARPETA_DOCUMENTOS.mkdir(
    parents=True,
    exist_ok=True
)

CARPETA_FAISS.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("Configuración general cargada.")


# ============================================================
# 4. CONFIGURAR API KEY DE GEMINI
# ============================================================

try:
    # Obtenemos la clave guardada en los secretos de Colab.
    GOOGLE_API_KEY = userdata.get(
        "GOOGLE_API_KEY"
    )

    if not GOOGLE_API_KEY:
        raise ValueError(
            "El secreto GOOGLE_API_KEY no existe o está vacío."
        )

    # También guardamos la clave como variable de entorno.
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

    # Creamos el cliente oficial de Google Gemini.
    cliente_gemini = genai.Client(
        api_key=GOOGLE_API_KEY
    )

    print("Cliente Gemini configurado correctamente.")

except Exception as error:
    raise RuntimeError(
        "No fue posible configurar Gemini.\n\n"
        "Comprueba que hayas creado un secreto en Colab llamado "
        "GOOGLE_API_KEY y que tenga habilitado el acceso al notebook."
    ) from error


# ============================================================
# 5. FUNCIONES PARA DETECTAR UN MODELO GEMINI DISPONIBLE
# ============================================================

def normalizar_nombre_modelo(nombre: str) -> str:
    """
    Elimina el prefijo 'models/' del nombre del modelo.

    Ejemplo:

        models/gemini-3-flash-preview

    se transforma en:

        gemini-3-flash-preview
    """

    if not nombre:
        return ""

    return str(nombre).replace(
        "models/",
        ""
    ).strip()


def es_modelo_textual(nombre: str) -> bool:
    """
    Determina si un modelo parece adecuado para generar texto.

    Se excluyen modelos destinados principalmente a:
    - Embeddings
    - Imágenes
    - Video
    - Audio
    - Voz
    """

    nombre_minuscula = nombre.lower()

    exclusiones = [
        "embedding",
        "imagen",
        "image",
        "veo",
        "tts",
        "audio",
        "speech",
        "aqa",
        "robotics"
    ]

    return not any(
        palabra in nombre_minuscula
        for palabra in exclusiones
    )


def obtener_metodos_modelo(modelo: Any) -> List[str]:
    """
    Obtiene los métodos o acciones soportadas por un modelo.

    La propiedad puede variar entre versiones del SDK,
    por eso se revisan diferentes nombres posibles.
    """

    metodos = []

    atributos_posibles = [
        "supported_actions",
        "supported_generation_methods"
    ]

    for atributo in atributos_posibles:
        valor = getattr(
            modelo,
            atributo,
            None
        )

        if valor:
            try:
                metodos.extend(
                    [str(elemento) for elemento in valor]
                )
            except TypeError:
                metodos.append(str(valor))

    return metodos


def modelo_admite_generacion(modelo: Any) -> bool:
    """
    Comprueba si los metadatos del modelo indican que admite
    generación de contenido.

    Si el SDK no informa métodos compatibles, se permite que
    el modelo continúe a la prueba práctica.
    """

    metodos = obtener_metodos_modelo(modelo)

    if not metodos:
        return True

    texto_metodos = " ".join(
        metodos
    ).lower()

    expresiones_validas = [
        "generatecontent",
        "generate_content",
        "generate content"
    ]

    return any(
        expresion in texto_metodos
        for expresion in expresiones_validas
    )


def listar_modelos_candidatos() -> List[str]:
    """
    Consulta los modelos disponibles para la API Key y devuelve
    posibles modelos Gemini compatibles con generación de texto.
    """

    candidatos = []

    try:
        for modelo in cliente_gemini.models.list():

            nombre_original = getattr(
                modelo,
                "name",
                ""
            )

            nombre = normalizar_nombre_modelo(
                nombre_original
            )

            if not nombre:
                continue

            if "gemini" not in nombre.lower():
                continue

            if not es_modelo_textual(nombre):
                continue

            if not modelo_admite_generacion(modelo):
                continue

            if nombre not in candidatos:
                candidatos.append(nombre)

    except Exception as error:
        print()
        print("No fue posible listar los modelos automáticamente.")
        print(
            f"Detalle: {type(error).__name__}: {error}"
        )

    return candidatos


def prioridad_modelo(nombre: str) -> Tuple[int, int, str]:
    """
    Asigna prioridad a los modelos.

    Preferencia general:
    1. Flash-Lite estable
    2. Flash estable
    3. Pro estable
    4. Modelos preview
    5. Otros modelos Gemini
    """

    nombre_minuscula = nombre.lower()

    es_preview = any(
        palabra in nombre_minuscula
        for palabra in [
            "preview",
            "experimental",
            "-exp",
            "latest"
        ]
    )

    if "flash-lite" in nombre_minuscula:
        categoria = 1
    elif "flash" in nombre_minuscula:
        categoria = 2
    elif "pro" in nombre_minuscula:
        categoria = 3
    else:
        categoria = 4

    preview_orden = 1 if es_preview else 0

    return (
        preview_orden,
        categoria,
        nombre
    )


def probar_modelo(nombre_modelo: str) -> bool:
    """
    Realiza una consulta mínima para comprobar que el modelo
    puede utilizarse realmente con la API Key.
    """

    try:
        respuesta = cliente_gemini.models.generate_content(
            model=nombre_modelo,
            contents=(
                "Responde únicamente con la palabra OK."
            ),
            config=types.GenerateContentConfig(
                temperature=0,
                max_output_tokens=20
            )
        )

        texto = getattr(
            respuesta,
            "text",
            ""
        )

        return bool(
            texto and texto.strip()
        )

    except Exception as error:
        print(
            f"Modelo descartado: {nombre_modelo}"
        )
        print(
            f"Motivo: {type(error).__name__}: {error}"
        )
        print()

        return False


def seleccionar_modelo_gemini() -> str:
    """
    Selecciona automáticamente un modelo Gemini disponible.

    Primero usa los modelos informados por la API.
    Si el listado falla, prueba algunos nombres conocidos como
    alternativa.
    """

    print()
    print("=" * 70)
    print("DETECCIÓN DEL MODELO GEMINI")
    print("=" * 70)

    candidatos = listar_modelos_candidatos()

    # Lista alternativa por si el endpoint de modelos no informa
    # correctamente todas las opciones disponibles.
    candidatos_respaldo = [
        "gemini-3.5-flash-lite",
        "gemini-3.6-flash",
        "gemini-3-flash-preview",
        "gemini-3.1-flash-lite-preview",
        "gemini-3.1-pro-preview"
    ]

    for modelo in candidatos_respaldo:
        if modelo not in candidatos:
            candidatos.append(modelo)

    candidatos = sorted(
        candidatos,
        key=prioridad_modelo
    )

    if not candidatos:
        raise RuntimeError(
            "No se encontraron modelos Gemini candidatos."
        )

    print("Modelos candidatos:")
    for modelo in candidatos:
        print("-", modelo)

    print()
    print("Comprobando cuáles funcionan con tu API Key...")
    print()

    for nombre_modelo in candidatos:

        if probar_modelo(nombre_modelo):
            print("=" * 70)
            print(
                "Modelo seleccionado:",
                nombre_modelo
            )
            print("=" * 70)

            return nombre_modelo

    raise RuntimeError(
        "No se encontró un modelo Gemini habilitado para esta "
        "API Key.\n\n"
        "Revisa el estado del proyecto en Google AI Studio o "
        "crea una API Key asociada a un proyecto nuevo."
    )


# Detectamos el modelo.
MODELO_GEMINI = seleccionar_modelo_gemini()


# ============================================================
# 6. SUBIR DOCUMENTOS
# ============================================================

print()
print("=" * 70)
print("CARGA DE DOCUMENTOS")
print("=" * 70)
print()
print(
    "Selecciona los archivos oficiales de Control Personal."
)
print(
    "Puedes cargar archivos DOCX y PDF."
)
print()

archivos_subidos = files.upload()

if not archivos_subidos:
    raise ValueError(
        "No se seleccionaron archivos."
    )

rutas_documentos = []

for nombre_archivo, contenido in archivos_subidos.items():

    extension = Path(
        nombre_archivo
    ).suffix.lower()

    if extension not in [
        ".docx",
        ".pdf"
    ]:
        print(
            f"Archivo ignorado por formato no compatible: "
            f"{nombre_archivo}"
        )
        continue

    ruta_destino = (
        CARPETA_DOCUMENTOS / nombre_archivo
    )

    with open(
        ruta_destino,
        "wb"
    ) as archivo_destino:
        archivo_destino.write(contenido)

    rutas_documentos.append(
        ruta_destino
    )

    print(
        f"Archivo guardado: {nombre_archivo}"
    )

if not rutas_documentos:
    raise ValueError(
        "No se cargaron archivos DOCX o PDF válidos."
    )

print()
print(
    f"Total de documentos válidos: "
    f"{len(rutas_documentos)}"
)


# ============================================================
# 7. LIMPIAR TEXTO
# ============================================================

def limpiar_texto(texto: str) -> str:
    """
    Limpia espacios, tabulaciones y saltos de línea excesivos.
    """

    if not texto:
        return ""

    # Reemplaza espacios y tabulaciones repetidas.
    texto = re.sub(
        r"[ \t]+",
        " ",
        texto
    )

    # Reduce grupos de tres o más saltos de línea.
    texto = re.sub(
        r"\n{3,}",
        "\n\n",
        texto
    )

    # Elimina espacios al inicio y al final de cada línea.
    lineas = [
        linea.strip()
        for linea in texto.splitlines()
    ]

    texto = "\n".join(lineas)

    return texto.strip()


# ============================================================
# 8. LEER DOCUMENTOS WORD Y PDF
# ============================================================

def cargar_documentos(
    rutas: List[Path]
) -> Tuple[List[Document], List[str]]:
    """
    Lee todos los documentos cargados.

    Para archivos DOCX utiliza Docx2txtLoader.
    Para archivos PDF utiliza PyPDFLoader.

    Retorna:
    - Lista de documentos procesados.
    - Lista de errores encontrados.
    """

    documentos_procesados = []
    errores = []

    for ruta in rutas:

        try:
            extension = ruta.suffix.lower()

            if extension == ".docx":
                loader = Docx2txtLoader(
                    str(ruta)
                )

            elif extension == ".pdf":
                loader = PyPDFLoader(
                    str(ruta)
                )

            else:
                continue

            documentos_archivo = loader.load()

            for numero_bloque, documento in enumerate(
                documentos_archivo,
                start=1
            ):

                contenido_limpio = limpiar_texto(
                    documento.page_content
                )

                if not contenido_limpio:
                    continue

                documento.page_content = (
                    contenido_limpio
                )

                documento.metadata["fuente"] = (
                    ruta.name
                )

                documento.metadata["tipo_archivo"] = (
                    extension
                )

                documento.metadata["bloque_original"] = (
                    numero_bloque
                )

                pagina_original = (
                    documento.metadata.get("page")
                )

                if pagina_original is not None:
                    documento.metadata["pagina"] = (
                        int(pagina_original) + 1
                    )

                documentos_procesados.append(
                    documento
                )

            print(
                f"Documento procesado: {ruta.name}"
            )

        except Exception as error:

            mensaje_error = (
                f"No fue posible procesar {ruta.name}: "
                f"{type(error).__name__}: {error}"
            )

            errores.append(
                mensaje_error
            )

            print(
                mensaje_error
            )

    return documentos_procesados, errores


documentos, errores_documentos = cargar_documentos(
    rutas_documentos
)

if not documentos:
    raise RuntimeError(
        "No fue posible extraer texto de los documentos."
    )

print()
print(
    f"Bloques documentales extraídos: {len(documentos)}"
)

if errores_documentos:
    print()
    print("Archivos con problemas:")

    for error in errores_documentos:
        print("-", error)


# ============================================================
# 9. MOSTRAR UNA VISTA PREVIA
# ============================================================

print()
print("=" * 70)
print("VISTA PREVIA DEL CONTENIDO")
print("=" * 70)

primer_documento = documentos[0]

print(
    "Fuente:",
    primer_documento.metadata.get(
        "fuente",
        "Sin identificar"
    )
)

print()
print(
    primer_documento.page_content[:1200]
)


# ============================================================
# 10. DIVIDIR DOCUMENTOS EN FRAGMENTOS
# ============================================================

divisor_texto = RecursiveCharacterTextSplitter(
    chunk_size=TAMANO_FRAGMENTO,
    chunk_overlap=SOLAPAMIENTO_FRAGMENTOS,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ],
    length_function=len
)

fragmentos = divisor_texto.split_documents(
    documentos
)

for numero_fragmento, fragmento in enumerate(
    fragmentos,
    start=1
):
    fragmento.metadata["fragmento_id"] = (
        numero_fragmento
    )

if not fragmentos:
    raise RuntimeError(
        "No fue posible dividir los documentos."
    )

print()
print("=" * 70)
print("DIVISIÓN DOCUMENTAL")
print("=" * 70)
print(
    f"Cantidad de fragmentos creados: "
    f"{len(fragmentos)}"
)

print()
print("Ejemplo del primer fragmento:")
print("-" * 70)
print(
    fragmentos[0].page_content[:1000]
)


# ============================================================
# 11. CONFIGURAR EMBEDDINGS LOCALES
# ============================================================

print()
print("=" * 70)
print("CARGA DEL MODELO DE EMBEDDINGS")
print("=" * 70)
print()
print(
    "La primera carga puede descargar el modelo desde "
    "Hugging Face."
)

try:
    embeddings = HuggingFaceEmbeddings(
        model_name=MODELO_EMBEDDINGS_LOCAL,
        model_kwargs={
            "device": "cpu"
        },
        encode_kwargs={
            "normalize_embeddings": True
        }
    )

    print()
    print(
        "Embeddings locales configurados correctamente."
    )

except Exception as error:
    raise RuntimeError(
        "No fue posible cargar el modelo local de embeddings.\n\n"
        f"Detalle: {type(error).__name__}: {error}"
    ) from error


# ============================================================
# 12. CREAR BASE VECTORIAL FAISS
# ============================================================

print()
print("=" * 70)
print("CREACIÓN DE LA BASE VECTORIAL")
print("=" * 70)
print()

try:
    # Eliminamos una base antigua para no mezclar contenido.
    if CARPETA_FAISS.exists():
        shutil.rmtree(
            CARPETA_FAISS
        )

    CARPETA_FAISS.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "Generando embeddings y construyendo FAISS..."
    )

    base_vectorial = FAISS.from_documents(
        documents=fragmentos,
        embedding=embeddings
    )

    base_vectorial.save_local(
        str(CARPETA_FAISS)
    )

    print(
        "Base vectorial creada correctamente."
    )

except Exception as error:

    print()
    print(traceback.format_exc())

    raise RuntimeError(
        "No fue posible crear la base vectorial FAISS.\n\n"
        f"Detalle: {type(error).__name__}: {error}"
    ) from error


# ============================================================
# 13. PROMPT DEL ASISTENTE
# ============================================================

PROMPT_SISTEMA = """
Eres el asistente oficial de soporte de Control Personal.

Control Personal es una plataforma SaaS orientada a la gestión
de personal, tickets, turnos, reemplazos, niveles de servicio
SLA, establecimientos, especialidades, reportes y procesos
operativos.

Tu función es ayudar a usuarios, técnicos, supervisores,
administradores y posibles clientes de Control Personal.

Debes cumplir obligatoriamente las siguientes reglas:

1. Responde utilizando solamente la información entregada en
   el contexto documental.

2. No inventes módulos, funcionalidades, precios, límites,
   permisos, procedimientos, políticas ni condiciones.

3. Si la información no aparece en el contexto, responde:

   "No encontré esta información en la documentación oficial
   de Control Personal. Te recomiendo contactar al administrador
   o al equipo de soporte."

4. Entrega respuestas claras, profesionales y fáciles de
   comprender.

5. Cuando expliques un procedimiento, utiliza pasos numerados.

6. Cuando la documentación lo indique, menciona qué rol tiene
   autorización para realizar la acción.

7. No compartas contraseñas, claves API, configuraciones privadas
   ni datos sensibles.

8. No respondas basándote en conocimientos externos.

9. No inventes precios si el documento no los especifica.

10. Si dos documentos presentan información diferente, indica
    que debe confirmarse con el administrador de la plataforma.

11. Responde siempre en español.

12. No agregues una sección de fuentes, porque el sistema la
    añadirá automáticamente.
"""


# ============================================================
# 14. FUNCIONES PARA MOSTRAR LAS FUENTES
# ============================================================

def describir_fuente(
    documento: Document
) -> str:
    """
    Crea un nombre legible para una fuente documental.
    """

    fuente = documento.metadata.get(
        "fuente",
        documento.metadata.get(
            "source",
            "Documento no identificado"
        )
    )

    pagina = documento.metadata.get(
        "pagina"
    )

    fragmento = documento.metadata.get(
        "fragmento_id"
    )

    descripcion = str(fuente)

    if pagina:
        descripcion += (
            f" — página {pagina}"
        )

    if fragmento:
        descripcion += (
            f" — fragmento {fragmento}"
        )

    return descripcion


# ============================================================
# 15. BUSCAR FRAGMENTOS RELEVANTES
# ============================================================

def buscar_fragmentos(
    pregunta: str,
    cantidad: int = CANTIDAD_RESULTADOS
) -> List[Tuple[Document, float]]:
    """
    Busca los fragmentos más similares a la pregunta.

    Devuelve:
    - Documento encontrado.
    - Distancia vectorial calculada por FAISS.
    """

    if not pregunta:
        return []

    pregunta = pregunta.strip()

    if not pregunta:
        return []

    resultados = (
        base_vectorial
        .similarity_search_with_score(
            query=pregunta,
            k=cantidad
        )
    )

    return resultados


# ============================================================
# 16. CONSTRUIR CONTEXTO DOCUMENTAL
# ============================================================

def construir_contexto(
    resultados: List[Tuple[Document, float]]
) -> Tuple[str, List[str], float]:
    """
    Une los fragmentos recuperados en un contexto para Gemini.

    Retorna:
    - Contexto completo.
    - Lista de fuentes.
    - Mejor distancia vectorial encontrada.
    """

    bloques_contexto = []
    fuentes = []
    distancias = []

    for numero, resultado in enumerate(
        resultados,
        start=1
    ):

        documento, distancia = resultado

        distancia = float(
            distancia
        )

        distancias.append(
            distancia
        )

        fuente = describir_fuente(
            documento
        )

        if fuente not in fuentes:
            fuentes.append(
                fuente
            )

        bloque = f"""
FRAGMENTO DOCUMENTAL {numero}

FUENTE:
{fuente}

CONTENIDO:
{documento.page_content}
""".strip()

        bloques_contexto.append(
            bloque
        )

    contexto = "\n\n" + (
        "\n\n"
        + "-" * 60
        + "\n\n"
    ).join(
        bloques_contexto
    )

    mejor_distancia = (
        min(distancias)
        if distancias
        else float("inf")
    )

    return (
        contexto,
        fuentes,
        mejor_distancia
    )


# ============================================================
# 17. FORMATEAR HISTORIAL DE GRADIO
# ============================================================

def extraer_texto_contenido(
    contenido: Any
) -> str:
    """
    Convierte diferentes formatos de contenido de Gradio
    en una cadena de texto.
    """

    if contenido is None:
        return ""

    if isinstance(contenido, str):
        return contenido

    if isinstance(contenido, dict):
        return str(
            contenido.get(
                "text",
                contenido.get(
                    "content",
                    contenido
                )
            )
        )

    return str(contenido)


def formatear_historial(
    historial: Any,
    limite: int = 8
) -> str:
    """
    Convierte el historial de Gradio en texto.

    Es compatible con:
    - Listas de diccionarios.
    - Pares pregunta-respuesta.
    - Objetos con atributos role y content.
    """

    if not historial:
        return "Sin conversación previa."

    lineas = []

    historial_reciente = historial[
        -limite:
    ]

    for elemento in historial_reciente:

        # Formato tipo diccionario:
        # {"role": "user", "content": "..."}
        if isinstance(elemento, dict):

            rol = str(
                elemento.get(
                    "role",
                    ""
                )
            ).lower()

            contenido = extraer_texto_contenido(
                elemento.get(
                    "content",
                    ""
                )
            )

            if rol == "user":
                lineas.append(
                    f"Usuario: {contenido}"
                )

            elif rol in [
                "assistant",
                "model"
            ]:
                lineas.append(
                    f"Asistente: {contenido}"
                )

        # Formato tradicional:
        # [pregunta, respuesta]
        elif (
            isinstance(elemento, (list, tuple))
            and len(elemento) >= 2
        ):

            pregunta_anterior = (
                extraer_texto_contenido(
                    elemento[0]
                )
            )

            respuesta_anterior = (
                extraer_texto_contenido(
                    elemento[1]
                )
            )

            if pregunta_anterior:
                lineas.append(
                    f"Usuario: {pregunta_anterior}"
                )

            if respuesta_anterior:
                lineas.append(
                    f"Asistente: {respuesta_anterior}"
                )

        # Formato de objeto ChatMessage.
        else:
            rol = getattr(
                elemento,
                "role",
                ""
            )

            contenido = getattr(
                elemento,
                "content",
                ""
            )

            contenido = extraer_texto_contenido(
                contenido
            )

            if rol == "user":
                lineas.append(
                    f"Usuario: {contenido}"
                )

            elif rol in [
                "assistant",
                "model"
            ]:
                lineas.append(
                    f"Asistente: {contenido}"
                )

    if not lineas:
        return "Sin conversación previa."

    return "\n".join(
        lineas
    )


# ============================================================
# 18. GENERAR RESPUESTA CON GEMINI
# ============================================================

def generar_respuesta_gemini(
    prompt: str
) -> str:
    """
    Envía el prompt al modelo Gemini seleccionado.
    """

    respuesta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            max_output_tokens=1200
        )
    )

    texto = getattr(
        respuesta,
        "text",
        None
    )

    if texto:
        return texto.strip()

    # Alternativa por si la respuesta no expone la propiedad text.
    candidatos = getattr(
        respuesta,
        "candidates",
        None
    )

    if candidatos:
        partes = []

        for candidato in candidatos:

            contenido = getattr(
                candidato,
                "content",
                None
            )

            if not contenido:
                continue

            partes_contenido = getattr(
                contenido,
                "parts",
                []
            )

            for parte in partes_contenido:

                texto_parte = getattr(
                    parte,
                    "text",
                    None
                )

                if texto_parte:
                    partes.append(
                        texto_parte
                    )

        if partes:
            return "\n".join(
                partes
            ).strip()

    raise RuntimeError(
        "Gemini no entregó una respuesta de texto."
    )


# ============================================================
# 19. FUNCIÓN PRINCIPAL DEL CHATBOT
# ============================================================

def responder_chatbot(
    mensaje: str,
    historial: Optional[Any] = None
) -> str:
    """
    Función principal conectada a Gradio.

    Flujo RAG:

    1. Recibe la pregunta.
    2. Busca fragmentos en FAISS.
    3. Comprueba la relevancia.
    4. Construye un contexto documental.
    5. Envía el contexto a Gemini.
    6. Añade las fuentes consultadas.
    """

    if not mensaje:
        return (
            "Escribe una pregunta relacionada con "
            "Control Personal."
        )

    pregunta = mensaje.strip()

    if not pregunta:
        return (
            "Escribe una pregunta relacionada con "
            "Control Personal."
        )

    try:
        # ----------------------------------------------------
        # Buscar los fragmentos más relevantes.
        # ----------------------------------------------------

        resultados = buscar_fragmentos(
            pregunta=pregunta,
            cantidad=CANTIDAD_RESULTADOS
        )

        if not resultados:
            return (
                "No encontré esta información en la "
                "documentación oficial de Control Personal. "
                "Te recomiendo contactar al administrador "
                "o al equipo de soporte."
            )

        # ----------------------------------------------------
        # Construir el contexto.
        # ----------------------------------------------------

        contexto, fuentes, mejor_distancia = (
            construir_contexto(
                resultados
            )
        )

        # ----------------------------------------------------
        # Comprobar que la pregunta tenga suficiente relación
        # con el contenido documental.
        # ----------------------------------------------------

        if mejor_distancia > UMBRAL_DISTANCIA:
            return (
                "No encontré esta información en la "
                "documentación oficial de Control Personal. "
                "Te recomiendo contactar al administrador "
                "o al equipo de soporte."
            )

        # ----------------------------------------------------
        # Preparar historial.
        # ----------------------------------------------------

        historial_formateado = (
            formatear_historial(
                historial
            )
        )

        # ----------------------------------------------------
        # Construir el prompt completo.
        # ----------------------------------------------------

        prompt_completo = f"""
{PROMPT_SISTEMA}

HISTORIAL RECIENTE DE LA CONVERSACIÓN:

{historial_formateado}


CONTEXTO RECUPERADO DESDE LOS DOCUMENTOS OFICIALES:

{contexto}


PREGUNTA ACTUAL DEL USUARIO:

{pregunta}


INSTRUCCIONES FINALES:

- Responde únicamente utilizando el contexto documental.
- No inventes información.
- No menciones puntajes ni distancias vectoriales.
- No incluyas una sección de fuentes.
- Si corresponde, entrega el procedimiento paso a paso.
""".strip()

        # ----------------------------------------------------
        # Obtener respuesta de Gemini.
        # ----------------------------------------------------

        respuesta_generada = (
            generar_respuesta_gemini(
                prompt_completo
            )
        )

        # ----------------------------------------------------
        # Preparar fuentes para la respuesta.
        # ----------------------------------------------------

        fuentes_markdown = "\n".join(
            f"- `{fuente}`"
            for fuente in fuentes
        )

        respuesta_final = f"""
{respuesta_generada}

---

### Fuentes documentales consultadas

{fuentes_markdown}
""".strip()

        return respuesta_final

    except Exception as error:

        print()
        print("=" * 70)
        print("ERROR DETALLADO DEL CHATBOT")
        print("=" * 70)
        print(
            traceback.format_exc()
        )

        return (
            "No fue posible procesar la consulta.\n\n"
            "Comprueba la conexión, el acceso al modelo Gemini "
            "y la cuota disponible.\n\n"
            f"**Detalle técnico:** "
            f"`{type(error).__name__}: {error}`"
        )


# ============================================================
# 20. REALIZAR UNA PRUEBA AUTOMÁTICA
# ============================================================

print()
print("=" * 70)
print("PRUEBA AUTOMÁTICA DEL CHATBOT")
print("=" * 70)

pregunta_prueba = (
    "¿Qué es Control Personal?"
)

print()
print("Pregunta:")
print(
    pregunta_prueba
)

print()
print("Respuesta:")
print(
    responder_chatbot(
        pregunta_prueba,
        historial=[]
    )
)


# ============================================================
# 21. CREAR INTERFAZ GRADIO
# ============================================================

ejemplos_chatbot = [
    "¿Qué es Control Personal?",
    "¿Cómo se crea un ticket?",
    "¿Qué prioridades tienen los tickets?",
    "¿Qué funciones tiene un supervisor?",
    "¿Cómo funciona el SLA?",
    "¿Cómo se asigna un reemplazo?",
    "¿Qué incluye el plan Enterprise?",
    "¿Cómo funciona el sistema multiestablecimiento?",
    "¿Cómo se protegen los datos personales?"
]

# Se utiliza la configuración mínima de ChatInterface para
# maximizar la compatibilidad entre versiones de Gradio.
interfaz = gr.ChatInterface(
    fn=responder_chatbot,
    title=(
        "Asistente de Soporte — Control Personal"
    ),
    description=(
        "Consulta información sobre tickets, roles, SLA, "
        "turnos, reemplazos, establecimientos, reportes, "
        "privacidad, planes y funcionamiento de Control Personal."
    ),
    examples=ejemplos_chatbot
)


# ============================================================
# 22. INICIAR EL CHATBOT
# ============================================================

print()
print("=" * 70)
print("INICIANDO INTERFAZ")
print("=" * 70)
print()
print(
    "Cuando aparezca el enlace público, haz clic para abrir "
    "el chatbot."
)

interfaz.launch(
    share=True,
    debug=True
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 20.1 MB/s eta 0:00:00
CHATBOT DE SOPORTE — CONTROL PERSONAL

Librerías importadas correctamente.
Versión de Gradio: 6.20.0

Configuración general cargada.
Cliente Gemini configurado correctamente.

DETECCIÓN DEL MODELO GEMINI
Modelos candidatos:
- gemini-2.0-flash-lite
- gemini-2.0-flash-lite-001
- gemini-2.5-flash-lite
- gemini-3.1-flash-lite
- gemini-3.5-flash-lite
- gemini-2.0-flash
- gemini-2.0-flash-001
- gemini-2.5-flash
- gemini-3.5-flash
- gemini-3.6-flash
- gemini-2.5-pro
- gemini-3.1-flash-lite-preview
- gemini-flash-lite-latest
- gemini-3-flash-preview
- gemini-3.1-flash-live-preview
- gemini-flash-latest
- gemini-omni-flash-preview
- gemini-3-pro-preview
- gemini-3.1-pro-preview
- gemini-3.1-pro-preview-customtools
- gemini-pro-latest
- gemini-2.5-computer-use-preview-10-2025
- gemini-3.5-live-translate-preview

Comprobando cuáles funcionan con tu API Key...

Modelo descartado: gemini-2.0-flash-lite
Motivo: ClientEr

Saving Base_de_Conocimiento_Control_Personal_v1_1.docx to Base_de_Conocimiento_Control_Personal_v1_1 (4).docx
Saving FAQ_Control_Personal_Profesional.docx to FAQ_Control_Personal_Profesional (4).docx
Saving Planes_y_Precios_Control_Personal_Profesional.docx to Planes_y_Precios_Control_Personal_Profesional (4).docx
Saving Politica_Privacidad_Control_Personal_Profesional.docx to Politica_Privacidad_Control_Personal_Profesional (4).docx
Saving Terminos_y_Condiciones_Control_Personal_Profesional.docx to Terminos_y_Condiciones_Control_Personal_Profesional (4).docx
Archivo guardado: Base_de_Conocimiento_Control_Personal_v1_1 (4).docx
Archivo guardado: FAQ_Control_Personal_Profesional (4).docx
Archivo guardado: Planes_y_Precios_Control_Personal_Profesional (4).docx
Archivo guardado: Politica_Privacidad_Control_Personal_Profesional (4).docx
Archivo guardado: Terminos_y_Condiciones_Control_Personal_Profesional (4).docx

Total de documentos válidos: 5
Documento procesado: Base_de_Conocimiento_Co

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embeddings locales configurados correctamente.

CREACIÓN DE LA BASE VECTORIAL

Generando embeddings y construyendo FAISS...
Base vectorial creada correctamente.

PRUEBA AUTOMÁTICA DEL CHATBOT

Pregunta:
¿Qué es Control Personal?

Respuesta:
Control Personal es una plataforma SaaS orientada a la gestión operativa de organizaciones que administran personal, mantenimiento e incidencias.

Su objetivo principal es centralizar la operación, mejorar el cumplimiento de los niveles de servicio (SLA), optimizar la asignación de recursos, mantener una trazabilidad completa y proporcionar indicadores para la toma de decisiones.

La plataforma permite gestionar tickets, turnos, reemplazos, usuarios, reportes e indicadores desde una única solución multiestablecimiento. Su arquitectura funcional está compuesta por una aplicación web, una PWA para dispositivos móviles, una API REST, una base de datos centralizada y un sistema de autenticación basado en roles.

---

### Fuentes documentales consultada